# Bootcamp QML Example Exercise

## Learning a Probability Distribution with a Variational Quantum Circuit

This exercise is inspired by the distribution-loading part of quantum option-pricing workflows. In those workflows, one often needs a quantum state whose measurement probabilities match a target probability distribution.

Instead of building a full qGAN with a generator and discriminator, this exercise uses a simpler supervised target:

$$
p_{\mathrm{target}}(j)
\approx
p_{\mathrm{QNN}}(j;\theta),
\qquad
j = 0,\ldots,2^n-1.
$$

Your task is to train a variational quantum circuit so that its computational-basis probabilities approximate a binned lognormal distribution.


## What You Should Achieve

By the end of the notebook, you should have:

- generated a lognormal sample,
- converted it into a 16-bin probability distribution,
- built a 4-qubit variational quantum circuit,
- returned probabilities with `qml.probs`,
- trained the circuit parameters with PyTorch,
- compared the learned quantum distribution against the target distribution.

Keep the implementation small. This is a state-preparation/distribution-fitting exercise, not a full generative adversarial model.


## Imports

Use the repository's existing `uv` environment. Do not add `pip install` cells.


In [ ]:
# Pseudo-code:
#   1. import NumPy, Matplotlib, PyTorch, and PennyLane
#   2. set random seeds
#   3. define the number of qubits and bins

# Setup: imports and exercise constants.
import time

import matplotlib.pyplot as plt
import numpy as np
import pennylane as qml
import torch

np.random.seed(7)
torch.manual_seed(7)

n_qubits = 4
n_bins = 2**n_qubits

print("PennyLane version:", qml.__version__)
print("PyTorch version:", torch.__version__)
print("number of bins:", n_bins)


## A Tiny PennyLane (`qml`) Probability Example

This exercise uses `qml.probs`, which returns the probabilities of measuring computational-basis states.

For two qubits, the output has four entries:

$$
[P(00), P(01), P(10), P(11)].
$$

The example below is only a syntax warm-up. It is not the assignment solution.


In [ ]:
# Pseudo-code:
#   1. create a two-qubit simulator
#   2. apply two rotation gates
#   3. return all computational-basis probabilities
#   4. verify that probabilities sum to one

# Minimal qml.probs example.
import numpy as np
import pennylane as qml

demo_dev = qml.device("default.qubit", wires=2)

@qml.qnode(demo_dev)
def probability_demo():
    qml.RY(0.6, wires=0)
    qml.RY(1.1, wires=1)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])

demo_probs = probability_demo()

print(qml.draw(probability_demo)())
print("probabilities [00, 01, 10, 11]:", np.round(demo_probs, 3))
print("sum:", np.sum(demo_probs))


## Target Data: Binned Lognormal Distribution

A quantum computer with \(n\) qubits has \(2^n\) computational-basis states. With 4 qubits, we can represent 16 discrete outcomes:

$$
|0000\rangle,\ |0001\rangle,\ \ldots,\ |1111\rangle.
$$

We will sample a lognormal distribution, discretize it into 16 bins, and use the normalized histogram as the target probability vector.


In [ ]:
# Pseudo-code:
#   1. sample a lognormal distribution
#   2. discretize samples into 16 bins
#   3. normalize counts into probabilities
#   4. plot the target distribution

# Target distribution construction.
import matplotlib.pyplot as plt
import numpy as np
import torch

n_samples = 10_000
mu = 1.0
sigma = 0.75

samples = np.random.lognormal(mean=mu, sigma=sigma, size=n_samples)
bin_indices = np.clip(np.floor(samples).astype(int), 0, n_bins - 1)

target_distribution = np.bincount(bin_indices, minlength=n_bins).astype("float32")
target_distribution = target_distribution / target_distribution.sum()
target_distribution_t = torch.tensor(target_distribution, dtype=torch.float32)

plt.figure(figsize=(7, 3.6))
plt.bar(range(n_bins), target_distribution, color="tab:blue", alpha=0.8)
plt.title("Target binned lognormal distribution")
plt.xlabel("basis state index")
plt.ylabel("probability")
plt.xticks(range(n_bins))
plt.grid(alpha=0.25, axis="y")
plt.tight_layout()
plt.show()

print("target distribution:", np.round(target_distribution, 3))
print("sum:", target_distribution.sum())


## Task 1 - Build the Variational Quantum Circuit

Define a 4-qubit circuit that returns

$$
p_{\mathrm{QNN}}(\theta)
=
\left[
P_\theta(0), P_\theta(1), \ldots, P_\theta(15)
\right].
$$

Requirements:

- use `qml.device("default.qubit", wires=n_qubits)`,
- define a `qml.qnode` with `interface="torch"`,
- include trainable rotations,
- include at least one entangling layer,
- return `qml.probs(wires=range(n_qubits))`,
- wrap the QNode with `qml.qnn.TorchLayer`.

You may use a dummy input vector of zeros. The trainable circuit parameters are what should learn the distribution.


In [ ]:
# TODO: implement the variational quantum circuit.
#
# Suggested structure:
#
# q_depth = ...
# dev = qml.device("default.qubit", wires=n_qubits)
#
# @qml.qnode(dev, interface="torch")
# def qnode(inputs, weights):
#     # optional fixed input rotations
#     # trainable rotation layers
#     # entangling gates
#     return qml.probs(wires=range(n_qubits))
#
# weight_shapes = {"weights": (...)}
# model = qml.qnn.TorchLayer(qnode, weight_shapes)


## Task 2 - Train the Circuit

Train the circuit so that its output probabilities match `target_distribution_t`.

A simple loss is mean squared error:

$$
L(\theta)
=
\frac{1}{2^n}
\sum_{j=0}^{2^n-1}
\left(
p_{\mathrm{QNN}}(j;\theta)
-
p_{\mathrm{target}}(j)
\right)^2.
$$

Use:

- `torch.optim.Adam`,
- a fixed dummy input such as `torch.zeros(n_qubits)`,
- a loss history list,
- a reasonable number of epochs.


In [ ]:
# TODO: train the quantum model.
#
# Suggested structure:
#
# optimizer = torch.optim.Adam(model.parameters(), lr=...)
# dummy_input = torch.zeros(n_qubits)
# loss_history = []
#
# for epoch in range(n_epochs):
#     optimizer.zero_grad()
#     predicted_distribution = model(dummy_input)
#     loss = ...
#     loss.backward()
#     optimizer.step()
#     loss_history.append(...)


## Task 3 - Visualize and Interpret

Compare the learned distribution against the target distribution.

Include:

- target distribution bar plot,
- learned distribution bar plot,
- training loss curve,
- final loss value,
- a short written comment.

In your comment, answer:

1. Did the circuit fit the target distribution?
2. Which bins were hardest to match?
3. How many quantum probabilities did the circuit have to learn?
4. Why is this a state-preparation problem rather than a full GAN?


In [ ]:
# TODO: visualize the result and write a short interpretation.
#
# Suggested structure:
#
# learned_distribution = model(dummy_input).detach().numpy()
# plot target_distribution and learned_distribution together
# plot loss_history
# print final loss and probability sum


## Submission Checklist

Before submitting, make sure your notebook contains:

- a target distribution plot,
- a completed PennyLane QNode,
- a wrapped `qml.qnn.TorchLayer`,
- a training loop,
- final target-vs-learned distribution plot,
- a loss curve,
- a short explanation of the result.
